# 01 — Análisis Exploratorio de Datos (EDA)

**TFM RunnAing** | Dataset: FitRec (Ni et al., 2019)

Objetivos:
- Cargar y describir el dataset
- Análisis de distribuciones (sesiones, usuarios, sexo, deporte)
- Análisis de span temporal por usuario (para identificar cohorte ACWR)
- Detección de valores ausentes y outliers
- Generar Tabla 4.1 (funnel de limpieza)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import auto_detect_and_load, filter_running_sessions

np.random.seed(42)
pd.set_option('display.max_columns', 20)
OUTPUTS = Path('../outputs/tables')
OUTPUTS.mkdir(parents=True, exist_ok=True)

print('Librerías cargadas correctamente')

: 

In [ ]:
# Carga del dataset completo
# Requiere: data/raw/fitrec.jsonl (ver README para descarga)
df_raw = auto_detect_and_load('../data/raw')
print(f'Sesiones cargadas: {len(df_raw):,}')
print(f'Columnas: {list(df_raw.columns)}')

In [ ]:
# Funnel de limpieza — Tabla 4.1
funnel = []
funnel.append({'paso': 'Dataset original', 'sesiones': len(df_raw),
               'usuarios': df_raw['userId'].nunique() if 'userId' in df_raw else None})

# Filtrar solo running
df_run = filter_running_sessions(df_raw)
funnel.append({'paso': 'Solo running (sport=run)', 'sesiones': len(df_run),
               'usuarios': df_run['userId'].nunique()})

# Filtrar sin gender
df_gender = df_run[df_run['gender'].notna()].copy()
funnel.append({'paso': 'Con género válido', 'sesiones': len(df_gender),
               'usuarios': df_gender['userId'].nunique()})

# Filtrar sesiones sin heart_rate (necesaria para calcular TRIMP)
df_hr = df_gender[df_gender['heart_rate'].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)].copy()
funnel.append({'paso': 'Con heart_rate (para TRIMP objetivo)', 'sesiones': len(df_hr),
               'usuarios': df_hr['userId'].nunique()})

# Filtrar sesiones sin GPS
df_gps = df_hr[df_hr['latitude'].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)].copy()
funnel.append({'paso': 'Con GPS completo (lat/lon/alt/timestamp)', 'sesiones': len(df_gps),
               'usuarios': df_gps['userId'].nunique()})

tabla_funnel = pd.DataFrame(funnel)
tabla_funnel['sesiones_eliminadas'] = tabla_funnel['sesiones'].shift(1) - tabla_funnel['sesiones']
print(tabla_funnel.to_string(index=False))

tabla_funnel.to_csv(OUTPUTS / 'tabla_4_1_funnel.csv', index=False)
print('\nTabla 4.1 guardada.')

In [ ]:
# Análisis de span temporal por usuario
if 'timestamp' in df_gps.columns:
    # Extraer primera y última fecha de la lista de timestamps
    df_gps['ts_min'] = df_gps['timestamp'].apply(
        lambda x: min(x)/1000 if max(x) > 1e9 else min(x)
    )
    df_gps['ts_max'] = df_gps['timestamp'].apply(
        lambda x: max(x)/1000 if max(x) > 1e9 else max(x)
    )
    df_gps['date'] = pd.to_datetime(df_gps['ts_min'], unit='s')

span_stats = df_gps.groupby('userId')['date'].agg(
    first_session='min',
    last_session='max',
    n_sessions='count'
).reset_index()
span_stats['span_days'] = (span_stats['last_session'] - span_stats['first_session']).dt.days

print('Distribución del span temporal por usuario:')
print(span_stats['span_days'].describe())

eligible_28 = (span_stats['span_days'] >= 28).sum()
print(f'\nUsuarios con span >= 28 días (aptos para ACWR): {eligible_28} '
      f'({100*eligible_28/len(span_stats):.1f}%)')

: 

In [ ]:
# Distribución por sexo
print('Distribución por sexo:')
print(df_gps['gender'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_gps['gender'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'])
axes[0].set_title('Sesiones por sexo')
axes[0].set_xlabel('Sexo')
axes[0].set_ylabel('Número de sesiones')

span_stats['span_days'].hist(bins=50, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].axvline(28, color='red', linestyle='--', label='Umbral 28 días (ACWR)')
axes[1].set_title('Distribución del span temporal por usuario')
axes[1].set_xlabel('Días')
axes[1].set_ylabel('Número de usuarios')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA completado.')